# Hybridisation number theory: counting the terms of the deranged determinant
Counts the terms of the power series of det(**D**) and adj(**D**), the derangement-matrix determinant and adjugate behind the Power Series H method, and checks closed forms (OEIS sequences: derangements, rencontres numbers, successive factorial differences, ...) against brute-force enumeration of the symmetric groups S_n: terms per order, positive vs. negative terms, terms containing given dimensions, and combinations of these.

Runtime: about 10-50 s by default. The brute-force counts stop at n = 7; set `FULL_RUN = True` in the first cell to enumerate up to n = 8 (and n = 10 in three scratchpad cells), which takes about 2 min. One cell draws terms as graphs and needs the optional `networkx`; without it that cell is skipped with a message.

# START

In [ ]:
import itertools
import numpy as np

from math import factorial, gcd, lcm
from fractions import Fraction

from sympy.combinatorics import Permutation
from sympy.combinatorics.named_groups import SymmetricGroup


# --- run settings ---------------------------------------------------------------------------------
# False (default): the brute-force "Counting" checks enumerate the symmetric groups S_n only up to
# n = N_COUNT_MAX = 7 (7! = 5,040 permutations), so the whole notebook runs in about a minute.
# True: they go up to the sizes used originally, n = 8 (40,320 permutations) and, in three cells,
# n = 10 (3,628,800 permutations; several GB of memory), which takes about 2 min.
# The closed-form ("Formula") rows always go up to n = 15.
FULL_RUN = False
N_COUNT_MAX = 7


def counting_N(N):
    """The largest n of a brute-force count: N in a full run, otherwise at most N_COUNT_MAX"""
    return N if FULL_RUN else min(N, N_COUNT_MAX)


# Binomial & nPk
Binomial a.k.a Pascal's triangle:

$n\mathcal{C}k \equiv \begin{pmatrix} n \\ k \end{pmatrix} \equiv \frac{n!}{(n-k)!k!}$

Permutation number:

$n\mathcal{P}k \equiv \frac{n!}{(n-k)!}$

In [ ]:
# https://oeis.org/A007318
def binomial(n, k):
    return round(factorial(n) / factorial(n - k) / factorial(k))

def nCk(n, k):
    return round(binomial(n, k))

# https://oeis.org/A008279
def nPk(n, k):
    return round(factorial(n) / factorial(n - k))

N = 15
print('Pascal`s triangle')
for n in range(N + 1):
    print(n, [binomial(n, k) for k in range(n + 1)])

print()
print('nPk triangle')
for n in range(N + 1):
    print(n, [nPk(n, k) for k in range(n + 1)])


# Factorials
Superfactorial:

$sf(n) \equiv \prod_{j=0}^n j!$

Alternating fractional factorial sum:

$E_n \equiv \sum_{j=0}^n \frac{(-1)^j}{j!}$

Running/cumulative alternating fractional factorial sum:

$F_n \equiv \sum_{j=0}^n E_j$

In [ ]:
# https://oeis.org/A000178
def superfactorial(n):
    result = 1
    for digit in range(n + 1):
        result *= factorial(digit)
    return round(result)

# https://oeis.org/A279046
def En_numerator(n):
    numerator_terms = [(-1)**j * superfactorial(n) / factorial(j) for j in range(n + 1)]
    result = sum(numerator_terms)
    return round(result)

def En_denominator(n):
    return round(superfactorial(n))

# final numerator = https://oeis.org/A053557
# final denominator = https://oeis.org/A053556
def En(n):
    return Fraction(
        numerator=En_numerator(n),
        denominator=En_denominator(n),
    )

# final numerator = E_{k+2} = https://math.stackexchange.com/questions/4917119/what-explains-these-numerators-being-the-same
# final denominator = https://oeis.org/A067653
def Fn(n):
    return sum([En(k) for k in range(n + 1)])

N = 15
print('superfactorial\n', [superfactorial(n) for n in range(N + 1)])

print()
print('En_numerator\n', [En_numerator(n) for n in range(N + 1)])

print()
print('En_denominator\n', [En_denominator(n) for n in range(N + 1)])

print()
print('En\n', [str(En(n)) for n in range(N + 1)])

print()
print('Fn\n', [str(Fn(n)) for n in range(N + 1)])

print(En(12))
Fn(10)

# Derangements

Full derangement:

$d_n = n! \times E_n$

Partial derangement of $f$ fixed points:

$d_{n,f} = \frac{n!}{f!} \times E_{n-f}$

Partial derangement of $(n-k)$ fixed points:

$d_{n, k} = n\mathcal{P}k \times E_k$


In [ ]:
# https://oeis.org/A000166
def dn(n):
    return round(factorial(n) * En(n))

# https://oeis.org/A008290
def dnf(n, f):
    return round(factorial(n) / factorial(f) * En(n - f))

# https://oeis.org/A098825
def dnk(n, k):
    return round(nPk(n, k) * En(k))

N = 15
print('Derangement numbers')
print([dn(n) for n in range(15)])

print()

print('Partial derangement triangle, dnf')
for n in range(N + 1):
    print([dnf(n, f) for f in range(n + 1)])

print()

print('Partial derangement triangle, dnk')
for n in range(N + 1):
    print([dnk(n, k) for k in range(n + 1)])


# Successive factorial differences
0th difference: $n! = (n - 0)! \times 1$

1st difference: $n! - (n - 1)! = (n - 1)! \times (n - 1)$

2nd difference: $(n-2)! \times (n^2 - 3n + 3)$

3rd difference: $(n-3)! \times (n^3 - 6n^2 + 14n -13)$

4th difference: $(n-4)! \times (n^4 - 10n^3 +41n^2 - 84n + 73)$

5th difference: $(n-5)! \times (n^5 - 15n^4 + 95n^3 - 325n^2 + 609n - 501)$

6th difference: $(n-6)! \times (n^6 - 21n^5 + 190n^4 - 965n^3 + 2944n^2 - 5155n + 4051)$

7th difference: $(n-7)! \times (n^7 - 28n^6 + 343n^5 - 2415n^4 + 10,689n^3 - 30,023n^2 + 49,790n - 37,633)$

8th difference: $(n-8)! \times (n^8 - 36n^7 + 574n^6 - 5348n^5 + 32,179n^4 - 129,220n^3 + 340,402n^2 - 539,616n + 394,353)$

9th difference: $(n-9)! \times (n^9 - 45n^8 + 906n^7 - 10,794n^6 + 84,567n^5 - 455,511n^4 + 1,698,374n^3 - 4,246,842n^2 + 6,478,521n - 4,596,553)$

$s^\text{th}$ difference: $(n - s)! \times N_s(n)$

where $[n^k]N_s(n) = (-1)^{s+k} \times \text{A216294}(s, k)$

and $N_n(n) = d_n$

In [ ]:
# https://oeis.org/A001563
def factorial_differences_1st(n):
    # coeffs: 1, -1
    return round(factorial(n - 1) * (n - 1))

# https://oeis.org/A001565
def factorial_differences_2nd(n):
    # coeffs: 1, -3, 3
    return round(factorial(n - 2) * (n**2 - 3*n + 3))

# https://oeis.org/A001565
def factorial_differences_3rd(n):
    # coeffs: 1, -6, 14, -13
    return round(factorial(n - 3) * (n**3 - 6*n**2 + 14*n - 13))

# https://oeis.org/A001688
def factorial_differences_4th(n):
    # coeffs 1, -10, 41, -84, 73
    return round(factorial(n - 4) * (n**4 - 10*n**3 + 41*n**2 - 84*n + 73))

# https://oeis.org/A001689
def factorial_differences_5th(n):
    # coeffs 1, -15, 95, -325, 609, -501
    return round(factorial(n - 5) * (n**5 - 15*n**4 + 95*n**3 - 325*n**2 + 609*n - 501))

# no OEIS yet...
def factorial_differences_6th(n):
    # coeffs 1, -21, 10, -965, 2944, -5155, 4051
    return round(factorial(n - 6) * (n**6 - 21*n**5 + 190*n**4 - 965*n**3 + 2944*n**2 - 5155*n + 4051))

# no OEIS yet...
def factorial_differences_7th(n):
    # coeffs 1, -28, 343, -2415, 10689, -30023, 49790, -37633
    return round(factorial(n - 7) * (n**7 - 28*n**6 + 343*n**5 - 2415*n**4 + 10_689*n**3 - 30_023*n**2 + 49_790*n - 37_633))

# no OEIS yet...
def factorial_differences_8th(n):
    # coeffs 1, -36, 574, -5348, 32179, -129220, 340402
    return round(factorial(n - 8) * (n**8 - 36*n**7 + 574*n**6 - 5348*n**5 + 32_179*n**4 - 129_220*n**3 + 340_402*n**2 - 539_616*n + 394_353))

# no OEIS yet...
def factorial_differences_9th(n):
    # coeffs 1, -45, 906, -10794, 84567, -455511, 1698374, -4246842, 6478521, -4596553
    return round(factorial(n - 8) * (n**9 - 45*n**8 + 906*n**7 - 10_794*n**6 + 84_567*n**5 - 455_511*n**4 + 1_698_374*n**3 - 4_246_842*n**2 + 6_478_521*n - 4_596_553))


# coeffs are: alternating ± https://oeis.org/A216294 !!
# T(n, k) = number of partial permutations of [n] that have k cycles ("bijection between 2 subsets")


# Each is a diagonal of Euler's difference table: https://oeis.org/A068106
N = 15
print('n!')
print([factorial(n) for n in range(N + 1)])

print()
print('1st factorial differences')
print([factorial_differences_1st(n) for n in range(1, N + 1)])

print()
print('2nd factorial differences')
print([factorial_differences_2nd(n) for n in range(2, N + 1)])

print()
print('3rd factorial differences')
print([factorial_differences_3rd(n) for n in range(3, N + 1)])

print()
print('4th factorial differences')
print([factorial_differences_4th(n) for n in range(4, N + 1)])

print()
print('5th factorial differences')
print([factorial_differences_5th(n) for n in range(5, N + 1)])

print()
print('6th factorial differences')
print([factorial_differences_6th(n) for n in range(6, N + 1)])

print()
print('7th factorial differences')
print([factorial_differences_7th(n) for n in range(7, N + 1)])

print()
print('8th factorial differences')
print([factorial_differences_8th(n) for n in range(8, N + 1)])

print()
print('9th factorial differences')
print([factorial_differences_9th(n) for n in range(9, N + 1)])


### Original formula: $\sum_{j=0}^s (-1)^j \times s\mathcal{C}j \times (n - j)!$
### Derangement formula: $\sum_{j=0}^{n-s} d_{n-j} \times \begin{pmatrix} n - s \\ j \end{pmatrix}$

In [ ]:
# https://oeis.org/A047920
def successive_factorial_differences(n, s):
    return round(sum([(-1)**j * binomial(s, j) * factorial(n - j) for j in range(s + 1)]))

def successive_factorial_differences_via_dn(n, s):
    return round(sum([dn(n - j) * binomial(n - s, j) for j in range(n - s + 1)]))

N = 15
print('Successive factorial differences triangle, original formula')
for n in range(N + 1):
    print([successive_factorial_differences(n, s) for s in range(n + 1)])

print()
print('Successive factorial differences triangle, dn formula')
for n in range(N + 1):
    print([successive_factorial_differences_via_dn(n, s) for s in range(n + 1)])


# Symmetric group: $\sigma \in S_n$

As assignments/arrangement numbers, e.g. $\sigma = 42135 \in S_5$
- $\sigma(1) = 4 \equiv 1 \to 4$
- $\sigma(2) = 2 \equiv 2 \to 2$
- $\sigma(3) = 1 \equiv 3 \to 1$
- $\sigma(4) = 3 \equiv 4 \to 3$
- $\sigma(5) = 5 \equiv 5 \to 5$

As cycles, e.g. $\sigma = \nu \varphi = (143)(2)(5) \in S_5$
- where $\nu$ is the non-fixed point part, a directed graph of cycles
    - $\nu = (143) \equiv (1 \to 4 \to 3 \to 1)$
- and $\varphi$ is the fixed points part, a directed graph of self-loops
    - $\varphi = (2)(5) \equiv (2 \to 2)(5 \to 5)$

Size of symmetric group $|S_n| = n!$

In [ ]:
def symmetric_group_sympy(n):
    if n == 0:
        return [None]

    # sorted lexicographically
    # af = array_form

    # SymmetricGroup(n) gives just the generators of the group... hence the need to generate the actual permutations
    return sorted(list(SymmetricGroup(n).generate_schreier_sims(af=True)))

_symmetric_group_cache = {}


def symmetric_group(n):
    """
    All n! permutations of 0, ..., n-1 in array form, sorted lexicographically

    NOTE: itertools.permutations yields exactly the sorted list that symmetric_group_sympy builds by
    Schreier-Sims generation (checked below for n <= 6), but about 100 times faster. Groups up to
    n = 8 are cached, since the counting cells below enumerate the same groups many times.
    """
    if n == 0:
        return [None]
    if n in _symmetric_group_cache:
        return _symmetric_group_cache[n]
    perms = [list(perm) for perm in itertools.permutations(range(n))]
    if n <= 8:
        _symmetric_group_cache[n] = perms
    return perms


assert all(symmetric_group(n) == symmetric_group_sympy(n) for n in range(7))

def symmetric_group_as_cycles(n, include_fixed_points=True):
    if n == 0:
        return [None]

    # full_cyclic_form includes fixed points
    if include_fixed_points:
        return [Permutation(perm).full_cyclic_form for perm in symmetric_group(n)]

    # cyclic_form excludes fixed points
    return [Permutation(perm).cyclic_form for perm in symmetric_group(n)]


N = 5
for n in range(N + 1):
    print(f'S{n}\n', symmetric_group(n))

print()
for n in range(N + 1):
    print(f'S{n} as cycles\n', symmetric_group_as_cycles(n))

print()
for n in range(N + 1):
    print(f'S{n} as cycles without fixed points\n', symmetric_group_as_cycles(n, include_fixed_points=False))


# Partial derangement sets
$D_{n,\alpha} \equiv \{\sigma \in S_n \forall f(\sigma) = n - \alpha\}$
- where $f(\sigma) \equiv \#\{\sigma(i) = i\}$ is the number of fixed points in a permutation

Key properties:
- The zeroth set just contains the identity permutation made of $n$ fixed points:
$D_{n,0} = \{\sigma_\text{id}\}$

- The first set contains no permutations:
$D_{n,1} = \empty$

- The last set contains all full derangements:
$D_{n,n} \equiv D_n$

- The size of each set defines the corresponding partial derangement number:
$d_{n,\alpha} \equiv |D_{n,\alpha}|$

- The union of all sets forms the symmetric group:
$\bigcup_{\alpha=0}^n D_{n,\alpha} = S_n$

In [ ]:
def partial_derangement_set(perms, num_fixed_points=0):
    if perms[0] == None:
        return [None]
    Dnf = []
    for perm in perms:
        f = 0
        for placement, item in enumerate(perm):
            if item == placement:
                f += 1
        if f == num_fixed_points:
            Dnf.append(perm)
    return Dnf

def fixed_points(n):
    if n == 0:
        return 1
    fp_seq = []
    for perm in symmetric_group(n):
        # this logic works only on non-cycle notation
        fp_seq.append(sum(placement == item for placement, item in enumerate(perm)))
    return fp_seq

def non_fixed_point_cycles(n):
    if n == 0:
        return 0
    count_even = 0
    count_odd = 0
    nfp_cycles_seq = []
    for perm in symmetric_group_as_cycles(n):
        # this logic works only on cycle notation
        nfp_cycles = sum(len(cycle) > 1 for cycle in perm)
        nfp_cycles_seq.append(nfp_cycles)
        if nfp_cycles % 2:
            count_odd += 1
        else:
            count_even += 1
    print(nfp_cycles_seq)
    return count_even, count_odd


N = counting_N(8)
print('Partial derangement triangle n vs. f')
for n in range(N + 1):
    Sn = symmetric_group(n)
    all_dnf = []
    for f in range(n + 1):
        Dnf = partial_derangement_set(Sn, f)
        all_dnf.append(len(Dnf))
    print(n, all_dnf)


print()
print('Partial derangement triangle n vs. a = n - a' )
for n in range(N + 1):
    Sn = symmetric_group(n)
    all_dna = []
    for a in range(n + 1):
        Dna = partial_derangement_set(Sn, n - a)
        all_dna.append(len(Dna))
    print(n, all_dna)

N = 6
print()
print('[fixed points per perm]')
for n in range(N + 1):
    print(n, fixed_points(n))


print()
print('[nfp cycles per perm]; (even, odd) nfp cycles')
for n in range(N + 1):
    print()
    print(n, non_fixed_point_cycles(n))


# Determinants
General determinant
$$
\mathbf{A}_n \equiv \begin{pmatrix} a_{11} & a_{12} & a_{13} & ... \\ a_{21} & a_{22} & a_{23} & ... \\ a_{31} & a_{32} & a_{33} & ... \\ ... & ... & ... & ... \end{pmatrix}
$$

$$
\det(\mathbf{A}_n) = \sum_{\sigma \in S_n} (-1)^m a_{1\sigma(1)}a_{2\sigma(2)}...a_{n\sigma(n)} = \sum_{\sigma \in S_n} (-1)^m a^n_\sigma
$$

- includes diagonal entries $a_{ii}$ from fixed points $\sigma(i) = i$
- $m$ is number of even-length cycles of $\sigma$, equivalent to the number of transpositions (swaps of 2 items) needed to define it

# Deranged determinant
$$
\mathbf{D}_n \equiv \begin{pmatrix} 1 & -R_{ab} & - R_{ac} & ... \\ -R_{ba} & 1 & -R_{bc} & ... \\ -R_{ca} & -R_{cb} & 1 & ... \\ ... & ... & ... & ... \end{pmatrix}
$$

$$
\det(\mathbf{D}_n)
=
\sum_{\alpha=0}^n \left(\sum_{\sigma \in D_{n,\alpha}}(-1)^c R^\alpha_\sigma \right)
=
1
- \sum_{\sigma \in D_{n,2}} R^2_\sigma
- \sum_{\sigma \in D_{n,3}} R^3_\sigma
+ \sum_{\sigma \in D_{n,4}}(-1)^c R^4_\sigma
+ ...
+ \sum_{\sigma \in D_{n}}(-1)^c R^n_\sigma
$$

- does not include diagonal entries since they are all equal to 1, hence deletes fixed points from cyclic structure: $\sigma = \nu\varphi \to \sigma = \nu\cancel{\varphi}$
- sign flip for each term included, hence $(-1)^m \to (-1)^{m+\alpha} = (-1)^c$
    - $c$ is the number of **non-fixed point cycles** in the cyclic structure of $\sigma$
        - $\alpha = 0$ case: $c=0$ for case of identity permutation of all fixed points $D_{n,0} = \{\sigma_\text{id}\}$, producing the $+1$ at the start
        - $\alpha = 1$ case: $D_{n,1} = \empty$, hence does not contribute any terms
        - $\alpha = 2, 3$ case: $c=1$ for all permtuations in $D_{n,2}$ and $D_{n,3}$, hence these terms are always negative 

In [ ]:
natural_numbers = '123456789'
alphabet = 'abcdefghijklmnopqrstuvwxyz'

def detA_term(perm):
    # for S0 case
    if perm is None:
        return '1'

    # (-1)**m
    even_length_cycles = sum((len(cycle) + 1) % 2 for cycle in Permutation(perm).full_cyclic_form)
    sign = '-' if even_length_cycles % 2 else '+'

    term = ''
    for placement, item in enumerate(perm):
        term = term + f'a{natural_numbers[placement]}{natural_numbers[item]}'
    return sign + term

def num_nontrivial_cycles(perm):
    """
    The number of cycles of length > 1 of a permutation in array form

    Equal to len(Permutation(perm).cyclic_form) (checked below), without building a sympy object.
    """
    seen, count = [False] * len(perm), 0
    for start in range(len(perm)):
        if seen[start] or perm[start] == start:
            continue
        count += 1
        position = start
        while not seen[position]:
            seen[position] = True
            position = perm[position]
    return count


_detD_term_cache = {}


def detD_term(perm):
    # for the S0 case
    if perm is None:
        return '1'

    # memoised: the counting cells below ask for the same terms many times
    key = tuple(perm)
    if key not in _detD_term_cache:
        _detD_term_cache[key] = _detD_term(perm)
    return _detD_term_cache[key]


def _detD_term(perm):
    # (-1)**c
    nfp_cycles = num_nontrivial_cycles(perm)
    sign = '-' if nfp_cycles % 2 else '+'

    term = ''
    for placement, item in enumerate(perm):
        if placement == item:
            continue
        term = term + f'R{alphabet[placement]}{alphabet[item]}'
    if term:
        return sign + term
    return '1'

# using these is really slow...
def all_detD_terms(n):
    return [detD_term(perm) for perm in symmetric_group(n)]

def detD_terms_of_order_a(n, a):
    # each term of order a is 1 + 3a characters long
    # 1 for the sign ±, (or for 1 itself)
    # + 3a, because each hybridisation function Rij / matrix entry aij is 3 characters long
    return [detD_term(perm) for perm in symmetric_group(n) if len(detD_term(perm))==1 + 3*a]

def positive_detD_terms(n):
    if n == 0:
        return ['1']
    return [detD_term(perm) for perm in symmetric_group(n) if not num_nontrivial_cycles(perm) % 2]

def negative_detD_terms(n):
    if n == 0:
        return []
    return [detD_term(perm) for perm in symmetric_group(n) if num_nontrivial_cycles(perm) % 2]

assert all(num_nontrivial_cycles(p) == len(Permutation(p).cyclic_form) for n in range(1, 7) for p in symmetric_group(n))

N = 4
for n in range(N + 1):
    print('\n')
    print(f'det(A{n})')
    for perm in symmetric_group(n):
        print(detA_term(perm))

    print()
    print(f'det(D{n})')
    for perm in symmetric_group(n):
        print(detD_term(perm))

print()
print('det(D4) breakdown')
print(detD_terms_of_order_a(4, 4))
print('positive')
print(positive_detD_terms(4))
print('negative')
print(negative_detD_terms(4))


In [ ]:
# Permutation(...).support() gives the items that are not singletons... i.e. the algebraic order of the detD term!
# full_cyclic_form includes fixed points, cyclic_form does not

perm2 = symmetric_group(2)[1]
print(
    perm2,
    Permutation(perm2).cyclic_form,
    Permutation(perm2).full_cyclic_form,
    len(Permutation(perm2).support()),
)

perm4 = symmetric_group(4)[6]
print(
    perm4,
    Permutation(perm4).cyclic_form,
    Permutation(perm4).full_cyclic_form,
    len(Permutation(perm4).support()),
)

perm5 = symmetric_group(5)[12]
print(
    perm5,
    Permutation(perm5).cyclic_form,
    Permutation(perm5).full_cyclic_form,
    len(Permutation(perm5).support()),
)


# Breaking down the deranged determinant...

$$
\det(\mathbf{D}_n) = \begin{vmatrix} 1 & -R_{ab} & - R_{ac} & ... \\ -R_{ba} & 1 & -R_{bc} & ... \\ -R_{ca} & -R_{cb} & 1 & ... \\ ... & ... & ... & ... \end{vmatrix} = \ ?
$$

Questions:
1. Terms of different order $\alpha$?
2. Positive vs. negative terms?
3. Terms with a given selection of dimensions?
4. Terms with a given selection of $R_{ij}$?

(Note: questions 1,2,3 and 1,2,4 are independent, but question 4 precludes question 3.)

### Specific answer #0: Terms per determinant $\det(\mathbf{D}_n)$
$\Longrightarrow T_n = n!$

where $n! = n\times(n - 1)\times(n-2) \times ... \times 2 \times 1 = 1, 1, 2, 6, 24, 120, 720, 5040, 40320, ...$

In [ ]:
N = 15
print('Formula')
print([factorial(n) for n in range(N + 1)])

print()

N = counting_N(10)
print('Counting')
print([len(symmetric_group(n)) for n in range(N + 1)])


### Specific answer #1: Terms of order $0 \leq \alpha \leq n$
$\Longrightarrow T_{n,\alpha} = d_{n,\alpha} = \tfrac{n!}{(n - \alpha)!} \times E_\alpha  = n\mathcal{P}\alpha \times E_\alpha$


where $E_n = \sum_{j=0}^n \tfrac{(-1)^j}{j!} = 1, 0, \tfrac{1}{2}, \tfrac{1}{3}, \tfrac{3}{8}, \tfrac{11}{30}, \tfrac{53}{144}, \tfrac{103}{280}, \tfrac{2119}{5760}... \longrightarrow e^{-1}$ as $n \to \infty$

In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [round(nPk(n, a) * En(a)) for a in range(n + 1)])

print()

N = counting_N(8)
print('Counting')
for n in range(N + 1):
    detD_terms = [detD_term(perm) for perm in symmetric_group(n)]
    print(n, [sum(len(term)==1 + 3*a for term in detD_terms) for a in range(n + 1)])


### Specific answer #2: Positive vs. negative terms
$\Longrightarrow T^\pm_n \equiv c^\pm_n = n! \pm 2^{n - 1}(2-n) = \tfrac{1}{2}\left(n! \pm \Delta_n\right)$

because
$c^+_n + c^-_n = n!$ must be true, and

$\Delta_n \equiv c^+_n - c^-_n = 2^{n - 1}(2-n)$ is also true, apparently! Although unclear why...

$\Delta_n = 1, 0, -4, -16, -48, -128, -320, -768, ...$

In [ ]:
N = 15
print('Formula')
print([int(0.5 * (factorial(n) + 2**(n - 1) * (2 - n))) for n in range(1, N + 1)])

print()

N = counting_N(8)
print('Counting')
print([len(positive_detD_terms(n)) for n in range(1, N + 1)])


In [ ]:
N = 15
print('Formula')
print([int(0.5 * (factorial(n) - 2**(n - 1) * (2 - n))) for n in range(1, N + 1)])

print()

N = counting_N(8)
print('Counting')
print([len(negative_detD_terms(n)) for n in range(1, N + 1)])


### Specific answer #3: Terms that include a number $0 \leq s \leq n$ of particular dimensions:
$\Longrightarrow T_n[s] = T_n[s-1] - T_{n - 1}[s-1]$

with $T_n[0] \equiv T_n = n!$

such that the final formula can be written as

$T_n[s] = \sum_{j=0}^s (-1)^j \times s\mathcal{C}j \times (n-j)! = E_s \times \sum_{j=0}^s s\mathcal{P}j \times (n - j)!$

where

$s\mathcal{C}j \equiv \begin{pmatrix} s \\ j \end{pmatrix} \equiv \tfrac{s!}{(s-j)!j!}$ is the binomial coefficient (Pascal's triangle)

or

$T_n[s] = \sum_{j=0}^{n-s} d_{n-j} \begin{pmatrix} n - s \\ j \end{pmatrix}$

where $d_n \equiv d_{n,n} = n! \times E_n = 1, 0, 1, 3, 9, 44, 265, 1854, 14833, ...$ are the derangement numbers

In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [factorial(n) - factorial(n-s) for s in range(n + 1)])

print()
N = counting_N(8)
print('Counting')
for n in range(N + 1):
    detD_terms = [detD_term(perm) for perm in symmetric_group(n)]
    Tns = []
    for s in range(n + 1):
        selection = 'abcdefghijklmnopqrstuvwxyz'[:s]
        Tns.append(sum(any([item in term for item in selection]) for term in detD_terms))
    print(Tns)

In [ ]:
for n in range(3, 10):
    print(factorial(n) - factorial(n - 3))

In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [successive_factorial_differences(n, s) for s in range(n + 1)])

print()
N = counting_N(8)
print('Counting')
for n in range(N + 1):
    detD_terms = [detD_term(perm) for perm in symmetric_group(n)]
    Tns = []
    for s in range(n + 1):
        selection = 'abcdefghijklmnopqrstuvwxyz'[:s]
        Tns.append(sum(all([item in term for item in selection]) for term in detD_terms))
    print(Tns)


# Combinations of answers #1, #2 and #3...

### #1 and #2: Positive vs. negative terms of order $0 \leq \alpha \leq n$
$\Longrightarrow T^\pm_{n,\alpha} \equiv c^\pm_{n,\alpha} = \tfrac{1}{2}(d_{n,\alpha} \pm \Delta_{n,\alpha}) = \tfrac{n\mathcal{P}\alpha}{2} \left(E_\alpha \pm \frac{1 - \alpha}{\alpha!} \right)$

because
- $c^+_{n,\alpha} + c^-_{n,\alpha} = d_{n,\alpha} = n\mathcal{P}\alpha \times E_\alpha$ must be true, and
- $\Delta_{n,\alpha} \equiv c^+_{n,\alpha} - c^-_{n,\alpha} = (1 - \alpha) \begin{pmatrix} n \\ \alpha \end{pmatrix} = \frac{1 - \alpha}{\alpha!} \times n\mathcal{P}\alpha$ is also true, apparently! Although unclear why...

In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [round(0.5 * nPk(n, a) * (float(En(a)) + (1 - a)/factorial(a))) for a in range(n + 1)])

print()

N = counting_N(8)
print('Counting')
for n in range(N + 1):
    if n == 0:
        positive_detD_terms = ['1']
    else:
        positive_detD_terms = [detD_term(perm) for perm in symmetric_group(n) if not num_nontrivial_cycles(perm) % 2]
    print(n, [sum(len(term) == 1 + 3*a for term in positive_detD_terms) for a in range(n + 1)])


In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [round(0.5 * nPk(n, a) * (float(En(a)) - (1 - a)/factorial(a))) for a in range(n + 1)])

print()

N = counting_N(8)
print('Counting')
for n in range(N + 1):
    if n == 0:
        negative_detD_terms = []
    else:
        negative_detD_terms = [detD_term(perm) for perm in symmetric_group(n) if num_nontrivial_cycles(perm) % 2]
    print(n, [sum(len(term) == 1 + 3*a for term in negative_detD_terms) for a in range(n + 1)])


### #2 and #3: Positive vs. negative terms that include a number $0 \leq s \leq n$ of particular dimensions

$\Longrightarrow T_n^\pm[s] = T_n^\pm[s-1] - T_{n - 1}^\pm[s-1]$

with $T_n^\pm[0] \equiv T_n^\pm = c_n^\pm = \frac{1}{2} \left(n! \pm \Delta_n\right)$

such that the final formula can be written as

$T_n^\pm[s] = \frac{1}{2} \left(T_n[s] \pm \Delta_n[s] \right)$

where

$\Delta_n[s] = \Delta_n[s-1] - \Delta_{n - 1}[s-1] = 2^{n-s-1}(2 - n - s)$

where

$\Delta_n[0] = \Delta_n = 2^{n - 1}(2 - n)$

In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [round(0.5 * (successive_factorial_differences(n, s) + 2**(n - s - 1) * (2 - n - s))) for s in range(n + 1)])

print()

N = counting_N(8)
print('Counting')
for n in range(N + 1):
    if n == 0:
        positive_detD_terms = ['1']
    else:
        positive_detD_terms = [detD_term(perm) for perm in symmetric_group(n) if not num_nontrivial_cycles(perm) % 2]
    Tn_plus_s = []
    for s in range(n + 1):
        selection = 'abcdefghijklmnopqrstuvwxyz'[:s]
        Tn_plus_s.append(sum(all([item in term for item in selection]) for term in positive_detD_terms))
    print(n, Tn_plus_s)

In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [round(0.5 * (successive_factorial_differences(n, s) - 2**(n - s - 1) * (2 - s - n))) for s in range(n + 1)])

print()

N = counting_N(8)
print('Counting')
for n in range(N + 1):
    if n == 0:
        negative_detD_terms = ['1']
    else:
        negative_detD_terms = [detD_term(perm) for perm in symmetric_group(n) if num_nontrivial_cycles(perm) % 2]
    Tn_minus_s = []
    for s in range(n + 1):
        selection = 'abcdefghijklmnopqrstuvwxyz'[:s]
        Tn_minus_s.append(sum(all([item in term for item in selection]) for term in negative_detD_terms))
    print(n, Tn_minus_s)

In [ ]:
N = 15
print('Formula')
for n in range(N + 1):
    print(n, [round(2**(n - s - 1) * (2 - s - n)) for s in range(n + 1)])

print()

### #1 and #3: Terms of order $s \leq \alpha \leq n$ that include a number $0 \leq s \leq n$ of particular dimensions

$\Longrightarrow T_{n,\alpha}[s] = T_{n,\alpha}[s-1] - T_{n - 1,\alpha}[s-1]$

with $T_{n,\alpha}[0] \equiv T_{n,\alpha} = d_{n,\alpha}$

such that the final formula can be written as

$T_{n,\alpha}[s] = d_{n,\alpha} \times \frac{\alpha\mathcal{P}s}{n\mathcal{P}s}= d_{n,\alpha} \times \frac{\alpha!}{n!} \frac{(n-s)!}{(\alpha-s)!}$

In [ ]:
N = 6
for s in range(N + 1):
    print()
    print(f'T(n, a)[s={s}]')
    print('Formula')
    for n in range(s, N + 1):
        print(n, [round(dnk(n, a) * nPk(a, s) / nPk(n, s)) for a in range(s, n + 1)])

    print()
    print('Counting')
    for n in range(s, N + 1):
        Tna_s = []
        for a in range(s, n + 1):
            order_a_terms = [detD_term(perm) for perm in symmetric_group(n) if len(detD_term(perm))==1 + 3*a]
            selection = 'abcdefghijklmnopqrstuvwxyz'[:s]
            Tna_s.append(sum(all([item in term for item in selection]) for term in order_a_terms))
        print(n, Tna_s)

N = 15
print()
print('More formula')
for s in range(N + 1):
    print()
    print(f'T(n, a)[s={s}]')
    for n in range(s, N + 1):
        print(n, [round(dnk(n, a) * nPk(a, s) / nPk(n, s)) for a in range(s, n + 1)])

### #1, #2 and #3: Positive vs. negative terms of order $s \leq \alpha \leq n$ that include a number $0 \leq s \leq n$ of particular items

$\Longrightarrow T^\pm_{n,\alpha}[s] = T^\pm_{n,\alpha}[s-1] - T^\pm_{n - 1,\alpha}[s]$

with $T^\pm_{n,\alpha}[0] \equiv T^\pm_{n,\alpha} = c^\pm_{n,\alpha}$

such that the final formula can be written as

$T^\pm_{n,\alpha}[s] = c^\pm_{n,\alpha} \times \frac{\alpha\mathcal{P}s}{n\mathcal{P}s}= c^\pm_{n,\alpha} \times \frac{\alpha!}{n!} \frac{(n-s)!}{(\alpha-s)!}$

In [ ]:
N = 6
for s in range(N + 1):
    print()
    print(f'T+(n, a)[s={s}]')
    print('Formula')
    for n in range(s, N + 1):
        print(n, [round(0.5 * nPk(n, a) * (float(En(a) + (1 - a)/factorial(a))) * nPk(a, s) / nPk(n, s)) for a in range(s, n + 1)])

    print()
    print('Counting')
    for n in range(s, N + 1):
        Tna_plus_s = []
        for a in range(s, n + 1):
            if n == 0:
                positive_order_a_terms = ['1']
            else:
                positive_order_a_terms = [detD_term(perm) for perm in symmetric_group(n) if len(detD_term(perm))==1 + 3*a and not num_nontrivial_cycles(perm) % 2]
            selection = 'abcdefghijklmnopqrstuvwxyz'[:s]
            Tna_plus_s.append(sum(all([item in term for item in selection]) for term in positive_order_a_terms))
        print(n, Tna_plus_s)

N = 15
print()
print('More formula')
for s in range(N + 1):
    print()
    print(f'T+(n, a)[s={s}]')
    for n in range(s, N + 1):
        print(n, [round(0.5 * nPk(n, a) * (float(En(a) + (1 - a)/factorial(a))) * nPk(a, s) / nPk(n, s)) for a in range(s, n + 1)])


In [ ]:
N = 6
for s in range(N + 1):
    print()
    print(f'T-(n, a)[s={s}]')
    print('Formula')
    for n in range(s, N + 1):
        print(n, [round(0.5 * nPk(n, a) * (float(En(a) - (1 - a)/factorial(a))) * nPk(a, s) / nPk(n, s)) for a in range(s, n + 1)])

    print()
    print('Counting')
    for n in range(s, N + 1):
        Tna_minus_s = []
        for a in range(s, n + 1):
            if n == 0:
                negative_order_a_terms = []
            else:
                negative_order_a_terms = [detD_term(perm) for perm in symmetric_group(n) if len(detD_term(perm))==1 + 3*a and num_nontrivial_cycles(perm) % 2]
            selection = 'abcdefghijklmnopqrstuvwxyz'[:s]
            Tna_minus_s.append(sum(all([item in term for item in selection]) for term in negative_order_a_terms))
        print(n, Tna_minus_s)

N = 15
print()
print('More formula')
for s in range(N + 1):
    print()
    print(f'T-(n, a)[s={s}]')
    for n in range(s, N + 1):
        print(n, [round(0.5 * nPk(n, a) * (float(En(a) - (1 - a)/factorial(a))) * nPk(a, s) / nPk(n, s)) for a in range(s, n + 1)])


# Specific answer #4: Terms that include a particular selection of hybridisation functions
a bit more complicated...

In [ ]:
# {num o: of all funcs Rij with i=a}
matching_origin = {
    0: [''],
    1: ['Rab'],
    2: ['Rab', 'Rac'],
    3: ['Rab', 'Rac', 'Rad'],
    4: ['Rab', 'Rac', 'Rad', 'Rae'],
    5: ['Rab', 'Rac', 'Rad', 'Rae', 'Raf'],
    6: ['Rab', 'Rac', 'Rad', 'Rae', 'Raf', 'Rag'],
    7: ['Rab', 'Rac', 'Rad', 'Rae', 'Raf', 'Rag', 'Rah'],
}

# {num e: of all funcs Rij with j=a}
matching_endpoint = {
    0: [''],
    1: ['Rba'],
    2: ['Rba', 'Rca'],
    3: ['Rba', 'Rca', 'Rda'],
    4: ['Rba', 'Rca', 'Rda', 'Rea'],
    5: ['Rba', 'Rca', 'Rda', 'Rea', 'Rfa'],
    6: ['Rba', 'Rca', 'Rda', 'Rea', 'Rfa', 'Rga'],
    7: ['Rba', 'Rca', 'Rda', 'Rea', 'Rfa', 'Rga', 'Rha'],
}

# {run length, r: run}
runs = {
    0: [''],
    1: ['Rab'],
    2: ['Rab', 'Rbc'],
    3: ['Rab', 'Rbc', 'Rcd'],
    4: ['Rab', 'Rbc', 'Rcd', 'Rde'],
    5: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref'],
    6: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfg'],
    7: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfg', 'Rgh'],
}

# {loop length, l: loop}
loops = {
    0: [''],
    1: [''],
    2: ['Rab', 'Rba'],
    3: ['Rab', 'Rbc', 'Rca'],
    4: ['Rab', 'Rbc', 'Rcd', 'Rda'],
    5: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Rea'],
    6: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfa'],
    7: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfg', 'Rga'],
    8: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfg', 'Rgh', 'Rha'],
}

non_matching = {
    0: [''],
    1: ['Rab'],
    2: ['Rab', 'Rcd'],
    3: ['Rab', 'Rcd', 'Ref'],
    4: ['Rab', 'Rcd', 'Ref', 'Rgh'],
}

In [ ]:
print('Tn ANY')

N = 7
print()
print('any o with matching origins from o=1 to (n - 1)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if any([func in detD_term(perm) for func in matching_origin[o]])])
            for o in range(1, n)
        ],
    )

print()
print('any e with matching endpoints from e=1 to (n - 1)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if any([func in detD_term(perm) for func in matching_endpoint[e]])])
            for e in range(1, n)
        ],
    )

print()
print('any runs of length r=1 to (n - 1)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if any([func in detD_term(perm) for func in runs[r]])])
            for r in range(1, n)
        ],
    )

print()
print('any loops of length l=2 to n')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if any([func in detD_term(perm) for func in loops[l]])])
            for l in range(2, n + 1)
        ],
    )

print()
print('any p non-matching node pairs p=1 to floor(n/2)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if any([func in detD_term(perm) for func in non_matching[m]])])
            for m in range(1, n // 2 + 1)
        ],
    )

print('\n\n')
print('Tn ALL')

print()
print('all o with matching origin from o=1 to (n - 1)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in matching_origin[o]])])
            for o in range(1, n)
        ],
    )

print()
print('all e matching endpoints from e=1 to (n - 1)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in matching_endpoint[e]])])
            for e in range(1, n)
        ],
    )

print()
print('all runs from length r=1 to (n - 1)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in runs[r]])])
            for r in range(1, n)
        ],
    )

print()
print('all loops of length l=2 to n')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in loops[l]])])
            for l in range(2, n + 1)
        ],
    )

print()
print('all p non-matching node pairs p=1 to floor(n/2)')
for n in range(N + 1):
    print(
        n,
        [
            len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in non_matching[p]])])
            for p in range(1, n // 2 + 1)
        ],
    )

# Off-diagonal adjugate entry $\text{adj}(\mathbf{D}_n)_{ij}$

### \#0: Total terms

$T_n = (n - 1)!$

### \#1: Terms per order

$T_{n,\alpha} = k_{n - 1,\alpha}$


##### determinant + path-cycles

$k_{n - 1,\alpha}= d_{n-2,\alpha-1} + d_{n - 1,\alpha}\times \frac{\alpha}{n - 1}$

##### falling factorial $\times$ running sum

$k_{n - 1,\alpha} = (n-2)_{\alpha-1} \times F_{\alpha-1}$

##### [OEIS A010027](https://oeis.org/A010027)
$k_{n - 1,\alpha} = \frac{1}{n - 1} \begin{pmatrix} n - 1 \\ \alpha \end{pmatrix} d_{\alpha + 1}$

### \#2: Positive vs. negative terms

$T_n^\pm = k_{n - 1}^\pm = \tfrac{1}{2}((n - 1)! \pm \nabla_{n - 1})$

where

$\nabla_{n - 1} \equiv 2^{n-2}$

### \#1 and \#2: Positive vs. negative terms per order

$T_{n,\alpha}^\pm = k_{n - 1,\alpha}^\pm = \tfrac{1}{2}(k_{n - 1,\alpha} \pm \nabla_{n - 1,\alpha})$

where

$\nabla_{n - 1,\alpha} \equiv \begin{pmatrix} n-2 \\ \alpha-1 \end{pmatrix}$

In [ ]:
dimension_letters = 'xyzuvw'
dimension_letters = alphabet

def sign(p):
    if p % 2:
        return '-'
    return '+'

def adjD_term(ij, perm):
    i, j = ij

    # for the adj(D1) case, that uses S0
    if perm is None:
        return '1'

    nfp_cycles = sum(len(cycle) > 1 for cycle in Permutation(perm).full_cyclic_form)
    term = ''
    if perm[0] == 0:
        nfp_cycles += 1
        term = f'R{ij}'

    origins = dimension_letters.replace(j, '')
    destinations = dimension_letters.replace(i, '')

    for placement, item in enumerate(perm):
        if placement == item:
            continue
        term = term + f'R{origins[placement]}{destinations[item]}'
    # (-1)**(c+1)
    return sign(nfp_cycles + 1) + term

N = counting_N(8)
ij = 'xy'
ij = 'ab'

for n in range(2, N + 1):
    pos, neg = 0, 0
    ord1, ord2, ord3, ord4, ord5, ord6, ord7, ord8 = 0, 0, 0, 0, 0, 0, 0, 0
    ord1_p, ord2_p, ord3_p, ord4_p, ord5_p, ord6_p, ord7_p, ord8_p = 0, 0, 0, 0, 0, 0, 0, 0
    ord1_m, ord2_m, ord3_m, ord4_m, ord5_m, ord6_m, ord7_m, ord8_m = 0, 0, 0, 0, 0, 0, 0, 0

    print('\n')
    print(f'adj(D{n})_{ij}')
    for perm in symmetric_group(n - 1):
        term = adjD_term(ij, perm)
        # print(term)
        if term[0] == '+':
            pos += 1
        else:
            neg += 1

        if len(term) == 1 + 3:
            ord1 += 1
            if term[0] == '+':
                ord1_p += 1
            else:
                ord1_m += 1

        elif len(term) == 1 + 6:
            ord2 += 1
            if term[0] == '+':
                ord2_p += 1
            else:
                ord2_m += 1

        elif len(term) == 1 + 9:
            ord3 += 1
            if term[0] == '+':
                ord3_p += 1
            else:
                ord3_m += 1

        elif len(term) == 1 + 12:
            ord4 += 1
            if term[0] == '+':
                ord4_p += 1
            else:
                ord4_m += 1

        elif len(term) == 1 + 15:
            ord5 += 1
            if term[0] == '+':
                ord5_p += 1
            else:
                ord5_m += 1

        elif len(term) == 1 + 18:
            ord6 += 1
            if term[0] == '+':
                ord6_p += 1
            else:
                ord6_m += 1

        elif len(term) == 1 + 21:
            ord7 += 1
            if term[0] == '+':
                ord7_p += 1
            else:
                ord7_m += 1

        elif len(term) == 1 + 24:
            ord8 += 1
            if term[0] == '+':
                ord8_p += 1
            else:
                ord8_m += 1

    print(f'+,-: {pos},{neg}')
    mine_p = round(0.5 * (factorial(n - 1) + 2**(n-2)))
    mine_m = round(0.5 * (factorial(n - 1) - 2**(n-2)))
    print(mine_p, mine_m)

    print('-------------')
    print(f'order: {ord1}, {ord2}, {ord3}, {ord4}, {ord5}, {ord6}, {ord7}')

    mine, mine2, oeis = [], [], []
    for a in range(1, n):
        mine.append(round(nPk(n - 2, a - 1) * Fn(a - 1)))
        mine2.append(round(dnk(n - 2, a - 1) + dnk(n - 1, a) * a / (n - 1)))
        oeis.append(round(1 / (n - 1) * binomial(n - 1, a) * dn(a + 1)))
    print(f'{mine=}')
    print(f'{mine2=}')
    print(f'{oeis=}')

    print('-------------')
    print(f'order +: {ord1_p}, {ord2_p}, {ord3_p}, {ord4_p}, {ord5_p}, {ord6_p}, {ord7_p}, {ord8_p}')
    print(f'order -: {ord1_m}, {ord2_m}, {ord3_m}, {ord4_m}, {ord5_m}, {ord6_m}, {ord7_m}, {ord8_m}')
    print(
        f'diff'
        f': {ord1_p - ord1_m}'
        f',{ord2_p - ord2_m}'
        f',{ord3_p - ord3_m}'
        f',{ord4_p - ord4_m}'
        f',{ord5_p - ord5_m}'
        f',{ord6_p - ord6_m}'
        f',{ord7_p - ord7_m}'
        f',{ord8_p - ord8_m}'
    )

    mine_ord_p = []
    mine_ord_m = []
    for a in range(1, n):
        oeis = round(1 / (n - 1) * binomial(n - 1, a) * dn(a + 1))
        mine_ord_p.append(round(0.5 * (oeis + binomial(n - 2, a - 1))))
        mine_ord_m.append(round(0.5 * (oeis - binomial(n - 2, a - 1))))

    print(mine_ord_p)
    print(mine_ord_m)


In [ ]:
N = 10
for n in range(n + 1):
    mine_ord_pm = []
    mine_ord_mm = []
    for a in range(1, n):
        oeis = round(1 / (n - 1) * binomial(n - 1, a) * dn(a + 1))
        # mine_ord_pm.append(round(0.5 * (oeis + binomial(n - 2, a - 1))))
        mine_ord_pm.append(round(0.5 * (1 / (n - 1)) * binomial(n - 1, a) * (dn(a + 1) + a)))

    print(mine_ord_pm)

for n in range(n + 1):
    mine_ord_pm = []
    mine_ord_mm = []
    for a in range(1, n):
        oeis = round(1 / (n - 1) * binomial(n - 1, a) * dn(a + 1))
        # mine_ord_pm.append(round(0.5 * (oeis - binomial(n - 2, a - 1))))
        mine_ord_pm.append(round(0.5 * (1 / (n - 1)) * binomial(n - 1, a) * (dn(a + 1) - a)))

    print(mine_ord_pm)


# Scratchpad / workings out

In [ ]:
len(SymmetricGroup(counting_N(10)).elements)

In [ ]:
# %%timeit
def num_fixed_points(perm):
    f = 0
    for cyc in perm.full_cyclic_form:
        if len(cyc) == 1:
            f += 1
    return f

# much quicker!
def num_fixed_points_2(perm):
    f = 0
    # for i, j in enumerate(perm.array_form):
    # even quicker!
    for i, j in enumerate(perm):
        if i == j:
            f += 1
    return f

n, a = counting_N(10), 2
Sn = SymmetricGroup(n)
Dna = list(filter(lambda perm: (num_fixed_points_2(perm) == (n - a)), Sn.elements))
# Dna_cyclic = [perm.full_cyclic_form for perm in Dna]
len(Dna)

In [ ]:
import itertools
all_triplets = []
n = 25
for triplet in list(itertools.combinations(alphabet[:n], 3)):
    all_triplets.append(triplet)
    fliplet = (triplet[0], triplet[2], triplet[1])
    all_triplets.append(fliplet)
len(all_triplets)

In [ ]:
import itertools
n = 10
alphabet = 'abcdefghijklmnopqrstuvwxyz'
all_pairs = list(itertools.combinations(alphabet[:n], 2))
len(all_pairs)

In [ ]:
# %%timeit
# networkx is optional: it is only needed here, to draw the terms as directed graphs
try:
    import networkx as nx
except ImportError:
    nx = None
    print('networkx is not installed, so this cell (which draws each term as a directed graph) is skipped;'
          ' install it with `pip install networkx` to run it')

if nx is not None:
    import matplotlib.pyplot as plt
    from copy import copy
    from collections import namedtuple

    Rn = namedtuple('Rn', ['sign', 'graph'])

    def Rn_str(sign, edges):
        if not edges:
            return '+1'
        if sign == 1:
            Rn = '+'
        else:
            Rn = '-'

        for edge in edges:
            Rn = f'{Rn}R{edge[0]}{edge[1]}.'
        return Rn

    abcdef = list('abcdef')
    numbers = list('0123456789')
    alphabet = list('abcdefghijklmnopqrstuvwyxz')

    one_particle_3D = list('xyz')
    two_particle_2D = ['x1', 'y1', 'x2', 'y2']
    two_particle_3D = ['x1', 'y1', 'z1', 'x2', 'y2', 'z2']

    dimension_labels = abcdef

    def generate_terms(
        n,
        i=None, j=None,
        dimension_labels=None,
        debug=False,
    ):
        """
        n = overall order of system
        i = selects diagonal element
        i,j = selects off-diagonal element
        """
        if dimension_labels is None:
            dimension_labels = list('abcdefghijklmnopqrstuvwyxz')
        dimension_labels = dimension_labels[:n]

        if i is None and j is None:
            det, diag = True, False
            Sn = SymmetricGroup(n)

            if debug:
                print(f'doing detD{n}')

        if i is not None:
            assert 0 <= i <= (n - 1)
            det, diag = False, True
            Sn = SymmetricGroup(n - 1)

            dim = dimension_labels[i]
            adj_ii_labels = copy(dimension_labels)
            adj_ii_labels.remove(dim)

            if debug:
                print(f'doing diagonal adjD{n}_{dim}{dim}')
                print(f'old labels {dimension_labels}')
                print(f'new labels {adj_ii_labels}')


        if i is not None and j is not None:
            assert j != i
            assert 0 <= i <= (n - 1)
            assert 0 <= j <= (n - 1)
            det, diag = False, False
            Sn = SymmetricGroup(n - 1)

            path_origin, path_destination = dimension_labels[i], dimension_labels[j]
            adj_ij_labels = copy(dimension_labels)
            adj_ij_labels.remove(path_origin)
            adj_ij_labels.remove(path_destination)
            adj_ij_labels.insert(0, path_origin + path_destination)

            if debug:
                print(f'doing off-diagonal adjD{n}_{path_origin}{path_destination}')
                print(f'old labels {dimension_labels}')
                print(f'new labels {adj_ij_labels}')


        # NOTE: could sort arrangements lexicographically? reduces performance by ~10%
        # arrangements = []
        # for perm in list(Sn.elements):
        #     arrangements.append(perm.array_form)
        # arrangements = sorted(arrangements)

        all_terms = []
        for nn, perm in enumerate(list(Sn.elements)):
            hybridigraph = nx.DiGraph()

            for position, label in enumerate(perm):

                # determinant and diagaonl adjugate follow same logic of deranged cycles
                if det or diag:
                    cycs = len(perm.cyclic_form)
                    sign = (-1)**cycs

                    # excludes fixed points
                    if position == label:
                        continue

                    if det:
                        labels = dimension_labels
                    elif diag:
                        labels = adj_ii_labels

                    origin = labels[position]
                    destination = labels[label]
                    hybridigraph.add_edge(origin, destination)

                    if debug:
                        print(f'edge {position}→{label} == {origin}→{destination}')


                # off-diagonal adjugate entry has path-cycles
                else:
                    cycs_without_0 = 0
                    for cyc in perm.cyclic_form:
                        if 0 not in cyc:
                            cycs_without_0 += 1
                    sign = (-1)**cycs_without_0

                    if position == 0 and label == 0:
                        hybridigraph.add_edge(path_origin, path_destination)
                        if debug:
                            print(f'edge {position}→{label} == {path_origin}→{path_destination}')

                    elif position == 0:
                        destination = adj_ij_labels[label]
                        hybridigraph.add_edge(path_origin, destination)
                        if debug:
                            print(f'edge {position}→{label} == {path_origin}→{destination}')

                    elif label == 0:
                        origin = adj_ij_labels[position]
                        hybridigraph.add_edge(origin, path_destination)
                        if debug:
                            print(f'edge {position}→{label} == {origin}→{path_destination}')

                    else:
                        # excludes fixed points
                        if position == label:
                            continue

                        origin = adj_ij_labels[position]
                        destination = adj_ij_labels[label]
                        hybridigraph.add_edge(origin, destination)
                        if debug:
                            print(f'edge {position}→{label} == {origin}→{destination}')

            if debug:
                print(perm.full_cyclic_form, Rn_str(sign, hybridigraph.edges))

            all_terms.append(Rn(sign=sign,graph=hybridigraph))

        return all_terms

    Dn_ij = generate_terms(3, 0, debug=True)

    for sign, term in Dn_ij:
        nx.draw_networkx(term)
        # pos = nx.spring_layout(term)
        # edge_labels = dict([((i, j), f'R_{i}{j}') for i, j in term.edges])
        # nx.draw_networkx_edge_labels(term, pos, edge_labels=edge_labels)
        plt.title(f'{sign=}')
        plt.show()


In [ ]:
N = 5
for n in range(n + 1):
    print([round(0.5 * (dnk(n, a) + (1 - a) * binomial(n, a))) for a in range(n + 1)])

In [ ]:
N = 5
for n in range(n + 1):
    print([round(dnk(n, a) / binomial(n, a)) for a in range(n + 1)])

In [ ]:
N = 10
print([round(0.5 * (dn(n) + (1 - n))) for n in range(n + 1)])
print([round(0.5 * (dn(n) - (1 - n))) for n in range(n + 1)])

In [ ]:
even, odd = 0, 0
for perm in symmetric_group_as_cycles(4):
    m = 0
    for cyc in perm:
        if not len(cyc) % 2:
            m += 1
    if m % 2:
        odd += 1
    else:
        even += 1
print(even, odd)

In [ ]:
N = counting_N(8)

for n in range(1, N + 1):
    # print(f'\n\n {n=}')

    even, odd = 0, 0
    even_per_ord = {a: 0 for a in range(n + 1)}
    odd_per_ord = {a: 0 for a in range(n + 1)}
    diff_per_ord = {a: 0 for a in range(n + 1)}

    for perm in symmetric_group(n):

        cyc_perm = Permutation(perm).full_cyclic_form
        f = sum(placement == item for placement, item in enumerate(perm))
        a = n - f
        # print(perm, cyc_perm, a, f)

        m = 0
        for cyc in cyc_perm:
            if not len(cyc) % 2:
                m += 1

        if not m % 2:
            even_per_ord[a] += 1
            even += 1
        else:
            odd_per_ord[a] += 1
            odd += 1

    diff_per_ord[a] = even_per_ord[a] - odd_per_ord[a]

    # print(even, odd)
    print(n, ':', list(even_per_ord.values()))
    # print(n, ':', list(odd_per_ord.values()))
    # print(n, ':', list(diff_per_ord.values()))


In [ ]:
N = 8
for n in range(1, N + 1):
    print(n, ':', [round(0.5 * binomial(n, a) * (dn(a) + (-1)**a * (1 - a))) for a in range(n + 1)])
    # print(n, ':', [(-1)**k * (1 - k) * binomial(n, k) for k in range(n + 1)])

In [ ]:
N = counting_N(8)

for n in range(1, N + 1):
    # print(f'\n\n {n=}')

    even, odd = 0, 0
    even_per_ord = {a: 0 for a in range(n + 1)}
    odd_per_ord = {a: 0 for a in range(n + 1)}
    diff_per_ord = {a: 0 for a in range(n + 1)}


    for perm in symmetric_group(n):

        cyc_perm = Permutation(perm).full_cyclic_form
        f = sum(placement == item for placement, item in enumerate(perm))
        a = n - f
        # print(perm, cyc_perm, a, f)

        m = 0
        for cyc in cyc_perm:
            if not len(cyc) % 2:
                m += 1

        if not m % 2:
            even_per_ord[a] += 1
            even += 1
        else:
            odd_per_ord[a] += 1
            odd += 1

    diff_per_ord[a] = even_per_ord[a] - odd_per_ord[a]

    # print(even, odd)
    # print(n, ':', list(even_per_ord.values()))
    print(n, ':', list(odd_per_ord.values()))
    # print(n, ':', list(diff_per_ord.values()))


In [ ]:
N = 8
for n in range(1, N + 1):
    print(n, ':', [round(0.5 * binomial(n, a) * (dn(a) - (-1)**a * (1 - a))) for a in range(n + 1)])

In [ ]:
N = counting_N(8)

for n in range(1, N + 1):
    # print(f'\n\n {n=}')

    even, odd = 0, 0
    even_per_ord = {a: 0 for a in range(n + 1)}
    odd_per_ord = {a: 0 for a in range(n + 1)}
    diff_per_ord = {a: 0 for a in range(n + 1)}


    for perm in symmetric_group(n):

        cyc_perm = Permutation(perm).full_cyclic_form
        f = sum(placement == item for placement, item in enumerate(perm))
        a = n - f
        # print(perm, cyc_perm, a, f)

        m = 0
        for cyc in cyc_perm:
            if not len(cyc) % 2:
                m += 1

        if not m % 2:
            even_per_ord[a] += 1
            even += 1
        else:
            odd_per_ord[a] += 1
            odd += 1

    for a in range(n + 1):
        diff_per_ord[a] = even_per_ord[a] - odd_per_ord[a]

    # print(even, odd)
    # print(n, ':', list(even_per_ord.values()))
    # print(n, ':', list(odd_per_ord.values()))
    print(n, ':', list(diff_per_ord.values()))


In [ ]:
N = 8
for n in range(1, N + 1):
    print(n, ':', [(-1)**k * (1 - k) * binomial(n, k) for k in range(n + 1)])

In [ ]:
dn(10)

In [ ]:
N = 10
print([round(0.5 * (dn(n) + (-1)**(n - 1)*(n - 1))) for n in range(N + 1)])
print([round(0.5 * (dn(n) - (-1)**(n - 1)*(n - 1))) for n in range(N + 1)])


In [ ]:
N = 100
print([round((-1)**(n - 1)*(n - 1)) for n in range(N + 1)])

In [ ]:
# any = all = (n - 1)!
legs = ['Rab']

# all = (n - 2)!
# any = a(n-2) for a(n) = (2n + 1) * n! = https://oeis.org/A007680
legs2 = ['Rab', 'Rcd']
legs2_chain = ['Rab', 'Rbc']

# all = (n - 3)!
# any = ???
legs3 = ['Rab', 'Rcd', 'Ref']
legs3_chain = ['Rab', 'Rbc', 'Rcd']


# all = (n - 4)!
# any = ???
legs4 = ['Rab', 'Rcd', 'Ref', 'Rgh']
legs4_chain = ['Rab', 'Rbc', 'Rcd', 'Rde']

filt, legs = any, legs4_chain
N = counting_N(8)
for n in range(1, N + 1):
    cc = 0
    for perm in symmetric_group(n):
        if filt([func in detD_term(perm) for func in legs]):
            cc += 1
    print(n, cc)
    # aa = [1 for perm in symmetric_group(n) if filt([func in detD_term(perm) for func in legs])]
    # print(n, len(aa))

In [ ]:
for n in range(2, 10):
    print(n, (2*(n-2) + 1) * factorial(n-2))


In [ ]:
print([(3*n - 1) * factorial(n) for n in range(1, 10)])

In [ ]:
legs_chain = ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfg']
legs = ['Rab', 'Rcd', 'Ref', 'Rgh', 'Rij', 'Rjk']

N = 7
filt, legs = any, legs_chain

print('h all any')
for n in range(1, N + 1):
    print(f'{n=}')
    for h in range(1, n):
        cc_any = 0
        cc_all = 0
        for perm in symmetric_group(n):
            if any([func in detD_term(perm) for func in legs[:h]]):
                cc_any += 1
            if all([func in detD_term(perm) for func in legs[:h]]):
                cc_all += 1
        print(h, cc_all, cc_any)

        # perms_all = [1 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in legs[:h + 1]])]
        # perms_any = [1 for perm in symmetric_group(n) if any([func in detD_term(perm) for func in legs[:h + 1]])]
        # print(h, len(perms_all), len(perms_any))

In [ ]:
all_chains = {
    0: [''],
    1: ['Rab'],
    2: ['Rab', 'Rbc'],
    3: ['Rab', 'Rbc', 'Rcd'],
    4: ['Rab', 'Rbc', 'Rcd', 'Rde'],
    5: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref'],
    6: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfg'],
    7: ['Rab', 'Rbc', 'Rcd', 'Rde', 'Ref', 'Rfg', 'Rgh'],
}

N = 7
filt, legs = any, legs

print('h all any')
for n in range(1, N + 1):
    print(f'{n=}')
    for h in range(n):
        cc_all = 0
        cc_any = 0
        for perm in symmetric_group(n):
            if all([func in detD_term(perm) for func in all_chains[h]]):
                cc_all += 1
            if any([func in detD_term(perm) for func in all_chains[h]]):
                cc_any += 1
        print(f'{h=}', cc_all, cc_any)

In [ ]:
print('Tn ALL')

print('all matching origin')
for n in range(N + 1):
    print(
        n,
        [
            (o, len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in matching_origin[o]])]))
            for o in range(n)
        ],
    )

print('all matching endpoint')
for n in range(N + 1):
    print(
        n,
        [
            (e, len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in matching_endpoint[e]])]))
            for e in range(n)
        ],
    )

print()
print('all runs')
for n in range(N + 1):
    print(
        n,
        [
            (r, len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in runs[r]])]))
            for r in range(n)
        ],
    )

print()
print('all loops')
for n in range(N + 1):
    print(
        n,
        [
            (l, len([0 for perm in symmetric_group(n) if all([func in detD_term(perm) for func in loops[l]])]))
            for l in range(n + 1)
        ],
    )

In [ ]:
for n in range(1, counting_N(8) + 1):
    aa = [detD_term(perm) for perm in symmetric_group(n) if any([func in detD_term(perm) for func in ['Rab', 'Rbc', 'Rcd']])]
    print(len(aa))

In [ ]:
list(filter(lambda perm: not num_nontrivial_cycles(perm) % 2, symmetric_group(4)))

In [ ]:
# c+ = n! + 2**(n - 1)*(2-n)
for n in range(1, counting_N(8) + 1):
    print(
        # cyclic_form excludes fixed point 'singletons'
        len([perm for perm in symmetric_group(n) if not num_nontrivial_cycles(perm) % 2]),
        int(0.5 * (factorial(n) + 2**(n - 1)*(2-n))),
    )

In [ ]:
# c- = n! - 2**(n - 1)*(2-n)
for n in range(1, counting_N(8) + 1):
    print(
        # cyclic_form excludes fixed point 'singletons'
        len([perm for perm in symmetric_group(n) if num_nontrivial_cycles(perm) % 2]),
        int(0.5 * (factorial(n) - 2**(n - 1)*(2-n))),
    )

$T_n[s] = T_n[s-1] - T_{n - 1}[s-1]$

$T_n[0] = n!$

In [ ]:
# number of terms with a given selection of dimensions
# T_n[s] = T_n[s-1] - T_n - 1[s-1]
# T_n[0] = n!
# e.g.
# T_6[3] = 426
n = 6
print(
    sum(
        (
            'a' in term and
            'b' in term and
            'e' in term
        )
    for term in [detD_term(perm) for perm in symmetric_group(n)]
))


for n in range(1, counting_N(8) + 1):
    detD_terms = [detD_term(perm) for perm in symmetric_group(n)]
    Tns = []
    for s in range(n + 1):
        selection = alphabet[:s]
        Tns.append(sum(all([item in term for item in selection]) for term in detD_terms))
    print(Tns)

$T_{n,\alpha}[s] = T_{n,\alpha}[s-1] - T_{n - 1,\alpha}[s-1]$

$T_{n,\alpha}[0] = d_{n,\alpha} = n\mathcal{P}\alpha \times E_\alpha$

In [ ]:
# number of terms of order α with a given selection of dimensions
# T_n,α[s] = T_n,α[s-1] - T_n - 1,α[s-1]
# T_n,α[0] = d_n,α

# each term of order α is 1 + 3α characters long
# 1 for the sign ±
# each hybridisation function Rij / matrix entry aij is 3 characters long, hence 3α

# e.g.
# T_7,6[3] = 1060
n, α = 7, 6
print(sum(
    (
        'a' in term and
        'b' in term and
        'd' in term and
        len(term)==1 + 3*α
    )
    for term in
    [detD_term(perm) for perm in symmetric_group(n)]
))


print('\n\n')
α = 3
print(f'T_n,{α}[s], triangle T(n, s)')
for n in range(1, 8):
    detD_terms = [detD_term(perm) for perm in symmetric_group(n)]
    Tnsα = []
    for s in range(n + 1):
        selection = alphabet[:s]
        Tnsα.append(
            sum(all(
                [item in term for item in selection] + [len(term)==1 + 3*α]) for term in detD_terms))
    print(Tnsα)

print('\n\n')
print('T(n, α) triangles for different s')
for s in range(n + 1):
    selection = alphabet[:s]
    print()
    print(f'T_n,α[{s}]')
    for n in range(1, 8):
        detD_terms = [detD_term(perm) for perm in symmetric_group(n)]
        Tnsα = []
        for α in range(n + 1):
            Tnsα.append(
                sum(all(
                    [item in term for item in selection] + [len(term)==1 + 3*α]) for term in detD_terms))
        print(Tnsα)

$T^\pm_n[s] = T^\pm_n[s-1] - T^\pm_{n - 1}[s-1]$

$T^\pm_n[0] = c^\pm_n = n! \pm 2^{n - 1}(2-n)$ 

In [ ]:
def Tpns(n, s):
    return successive_factorial_differences(n, s) + int(2**(n - s - 1) * (s + 2 - n))

for n in range(10):
    print(n, [Tpns(n, s) for s in range(n + 1)])

In [ ]:
# number of positive vs. negative terms with a given selection of dimensions
# T±_n[s] = T±_n[s-1] - T±_n - 1[s-1]
# T±_n[0] = c±_n


n = 6
positive_perms = [p for p in symmetric_group(n) if not num_nontrivial_cycles(p) % 2]
negative_perms = [p for p in symmetric_group(n) if num_nontrivial_cycles(p) % 2]

# e.g.
# T+_6[3] = 199
# T-_6[3] = 227
# s.t. 199 + 227 = 426 = T_6[3]
print(sum(
        (
            'a' in term and
            'b' in term and
            'd' in term
        )
        for term in
        [detD_term(perm) for perm in positive_perms]
    ),
    sum(
        (
            'a' in term and
            'b' in term and
            'd' in term
        )
        for term in
        [detD_term(perm) for perm in negative_perms]
    ),
)


print('\n\n')
print('T+(n, α) triangle')
for s in range(n + 1):
    print()
    print(f'T+_n,α[{s}]')
    for n in range(1, counting_N(8) + 1):
        positive_perms = [p for p in symmetric_group(n) if not num_nontrivial_cycles(p) % 2]
        detD_terms = [detD_term(perm) for perm in positive_perms]
        Tplus_nsα = []
        for α in range(1, n + 1):
            selection = alphabet[:s]
            Tplus_nsα.append(
                sum(all(
                    [item in term for item in selection] + [len(term)==1 + 3*α]) for term in detD_terms))
        print(Tplus_nsα)

print('\n\n')
print('T-(n, α) triangle')
for s in range(n + 1):
    print()
    print(f'T-_n,α[{s}]')
    for n in range(1, counting_N(8) + 1):
        negative_perms = [p for p in symmetric_group(n) if num_nontrivial_cycles(p) % 2]
        detD_terms = [detD_term(perm) for perm in negative_perms]
        Tminus_nsα = []
        for α in range(1, n + 1):
            selection = alphabet[:s]
            Tminus_nsα.append(
                sum(all(
                    [item in term for item in selection] + [len(term)==1 + 3*α]) for term in detD_terms))
        print(Tminus_nsα)

In [ ]:
# number of positive vs. negative terms of order α
# c±_n,α
# c+_n,α + c-_n,α = d_n,α

n, α = 7, 6
positive_perms = [p for p in symmetric_group(n) if not num_nontrivial_cycles(p) % 2]
negative_perms = [p for p in symmetric_group(n) if num_nontrivial_cycles(p) % 2]

# e.g.
# c+_7,6 = 910
# c-_7,6 = 945
# s.t. 910 + 945 = 1855 = d_7,6
print(sum(
        (
            len(term)==1 + 3*α
        )
        for term in
        [detD_term(perm) for perm in positive_perms]
    ),
    sum(
        (
            len(term)==1 + 3*α
        )
        for term in
        [detD_term(perm) for perm in negative_perms]
    ),
)

In [ ]:
# number of positive vs. negative terms of order α with a given selection of dimensions
# T±_n,α[0] = c±_n,α
# T±_n,α[s] = T_n,α[s-1] - T_n - 1,α[s-1]

# each term of order α is 1 + 3α characters long
# 1 for the sign ±
# each hybridisation function Rij / matrix entry aij is 3 characters long, hence 3α

n, α = 7, 6
positive_perms = [p for p in symmetric_group(n) if not num_nontrivial_cycles(p) % 2]
negative_perms = [p for p in symmetric_group(n) if num_nontrivial_cycles(p) % 2]

# e.g.
# T+7,6[3]] = 520
# T-_7,6[3] = 540
# s.t. 520 + 540 = 1060 = T_7,6[3]
print(sum(
        (
            'a' in term and
            'b' in term and
            'd' in term and
            len(term)==1 + 3*α
        )
        for term in
        [detD_term(perm) for perm in positive_perms]
    ),
    sum(
        (
            'a' in term and
            'b' in term and
            'd' in term and
            len(term)==1 + 3*α
        )
        for term in
        [detD_term(perm) for perm in negative_perms]
    ),
)


# END